In [2]:
import os
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel ,Field
from langchain.schema.runnable import RunnableParallel,RunnableBranch,RunnableLambda
from typing import Literal
load_dotenv()

True

In [4]:
import os
# Model
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.environ["GEMINI_API_KEY"]
)

In [5]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.8 MB 4.8 MB/s eta 0:00:04
   ---- ----------------------------------- 2.1/19.8 MB 5.3 MB/s eta 0:00:04
   ------ --------------------------------- 3.4/19.8 MB 5.6 MB/s eta 0:00:03
   ----------- ---------------------------- 5.5/19.8 MB 6.7 MB/s eta 0:00:03
   ---------------- ----------------------- 8.4/19.8 MB 8.3 MB/s eta 0:00:02
   ----------------------- ---------------- 11.5/19.8 MB 9.5 MB/s eta 0:00:01
   ------------------------------ --------- 14.9/19.8 MB 10.6 MB


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
model = ChatOpenAI(model="gpt-4o-mini")


In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma

In [9]:
from langchain.schema import Document

# Create LangChain documents for F1 drivers

doc1 = Document(
    page_content="Lewis Hamilton is a seven-time Formula 1 World Champion driving for Mercedes. Known for his aggressive yet smooth driving style, he holds numerous F1 records including wins and pole positions.",
    metadata={"team": "Mercedes"}
)

doc2 = Document(
    page_content="Max Verstappen, driving for Red Bull Racing, is a two-time F1 World Champion. He is known for his daring overtakes and consistency under pressure.",
    metadata={"team": "Red Bull Racing"}
)

doc3 = Document(
    page_content="Charles Leclerc is a talented driver for Ferrari. Known for his qualifying speed and precision on the track, he has shown great promise in multiple seasons.",
    metadata={"team": "Ferrari"}
)

doc4 = Document(
    page_content="George Russell drives for Mercedes alongside Lewis Hamilton. He is recognized for his clean racecraft, adaptability, and strong qualifying performances.",
    metadata={"team": "Mercedes"}
)

doc5 = Document(
    page_content="Sergio Perez, also known as Checo, races for Red Bull Racing. He is known for tire management, strategic racing, and supporting his teammate in championship pursuits.",
    metadata={"team": "Red Bull Racing"}
)


In [11]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [12]:
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

C:\Users\karth\AppData\Local\Temp\ipykernel_18540\697003433.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(


In [13]:
# add documents
vector_store.add_documents(docs)

['712fd7ea-6427-43b1-9c8c-fa28543237b4',
 '65efec92-aa6a-47da-88a8-bd22e8ce4bf3',
 '60ae2fa9-198c-4765-a3ee-cddd8f4e3aab',
 'd03c4aca-fe2c-49d9-a1aa-6d9c019ab8d8',
 '03b0ccc1-cf04-480f-808c-311a7f86bc1f']

In [14]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['712fd7ea-6427-43b1-9c8c-fa28543237b4',
  '65efec92-aa6a-47da-88a8-bd22e8ce4bf3',
  '60ae2fa9-198c-4765-a3ee-cddd8f4e3aab',
  'd03c4aca-fe2c-49d9-a1aa-6d9c019ab8d8',
  '03b0ccc1-cf04-480f-808c-311a7f86bc1f'],
 'embeddings': array([[ 0.00970307,  0.01926887, -0.01789609, ..., -0.01808329,
          0.01133793, -0.00575945],
        [ 0.00639277,  0.00776447,  0.0122241 , ..., -0.01434227,
         -0.00157188, -0.00245471],
        [ 0.0112207 ,  0.01709944,  0.01254649, ..., -0.00715926,
          0.0008658 , -0.00363137],
        [ 0.01283268,  0.00164395, -0.00282867, ..., -0.01192061,
          0.00334211, -0.00843181],
        [-0.00764074,  0.00923527,  0.01577611, ...,  0.00128051,
          0.00232672, -0.00501139]], shape=(5, 1536)),
 'documents': ['Lewis Hamilton is a seven-time Formula 1 World Champion driving for Mercedes. Known for his aggressive yet smooth driving style, he holds numerous F1 records including wins and pole positions.',
  'Max Verstappen, driving f

In [15]:
# search documents
vector_store.similarity_search(
    query='Who among are from Mercedes ?',
    k=2
)

[Document(metadata={'team': 'Mercedes'}, page_content='George Russell drives for Mercedes alongside Lewis Hamilton. He is recognized for his clean racecraft, adaptability, and strong qualifying performances.'),
 Document(metadata={'team': 'Mercedes'}, page_content='Lewis Hamilton is a seven-time Formula 1 World Champion driving for Mercedes. Known for his aggressive yet smooth driving style, he holds numerous F1 records including wins and pole positions.')]

In [16]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among are from Mercedes ?',
    k=2
)

[(Document(metadata={'team': 'Mercedes'}, page_content='George Russell drives for Mercedes alongside Lewis Hamilton. He is recognized for his clean racecraft, adaptability, and strong qualifying performances.'),
  0.3780439794063568),
 (Document(metadata={'team': 'Mercedes'}, page_content='Lewis Hamilton is a seven-time Formula 1 World Champion driving for Mercedes. Known for his aggressive yet smooth driving style, he holds numerous F1 records including wins and pole positions.'),
  0.3958805799484253)]

In [17]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Red Bull Racing"}
)

[(Document(metadata={'team': 'Red Bull Racing'}, page_content='Max Verstappen, driving for Red Bull Racing, is a two-time F1 World Champion. He is known for his daring overtakes and consistency under pressure.'),
  0.6231498718261719),
 (Document(metadata={'team': 'Red Bull Racing'}, page_content='Sergio Perez, also known as Checo, races for Red Bull Racing. He is known for tire management, strategic racing, and supporting his teammate in championship pursuits.'),
  0.634168267250061)]

In [18]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='712fd7ea-6427-43b1-9c8c-fa28543237b4', document=updated_doc1)


In [19]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['712fd7ea-6427-43b1-9c8c-fa28543237b4',
  '65efec92-aa6a-47da-88a8-bd22e8ce4bf3',
  '60ae2fa9-198c-4765-a3ee-cddd8f4e3aab',
  'd03c4aca-fe2c-49d9-a1aa-6d9c019ab8d8',
  '03b0ccc1-cf04-480f-808c-311a7f86bc1f'],
 'embeddings': array([[-0.00545721, -0.01906604,  0.00708297, ..., -0.01629019,
         -0.00041194,  0.00727194],
        [ 0.00639277,  0.00776447,  0.0122241 , ..., -0.01434227,
         -0.00157188, -0.00245471],
        [ 0.0112207 ,  0.01709944,  0.01254649, ..., -0.00715926,
          0.0008658 , -0.00363137],
        [ 0.01283268,  0.00164395, -0.00282867, ..., -0.01192061,
          0.00334211, -0.00843181],
        [-0.00764074,  0.00923527,  0.01577611, ...,  0.00128051,
          0.00232672, -0.00501139]], shape=(5, 1536)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple c

In [20]:
# delete document
vector_store.delete(ids=['712fd7ea-6427-43b1-9c8c-fa28543237b4'])

In [21]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['65efec92-aa6a-47da-88a8-bd22e8ce4bf3',
  '60ae2fa9-198c-4765-a3ee-cddd8f4e3aab',
  'd03c4aca-fe2c-49d9-a1aa-6d9c019ab8d8',
  '03b0ccc1-cf04-480f-808c-311a7f86bc1f'],
 'embeddings': array([[ 0.00639277,  0.00776447,  0.0122241 , ..., -0.01434227,
         -0.00157188, -0.00245471],
        [ 0.0112207 ,  0.01709944,  0.01254649, ..., -0.00715926,
          0.0008658 , -0.00363137],
        [ 0.01283268,  0.00164395, -0.00282867, ..., -0.01192061,
          0.00334211, -0.00843181],
        [-0.00764074,  0.00923527,  0.01577611, ...,  0.00128051,
          0.00232672, -0.00501139]], shape=(4, 1536)),
 'documents': ['Max Verstappen, driving for Red Bull Racing, is a two-time F1 World Champion. He is known for his daring overtakes and consistency under pressure.',
  'Charles Leclerc is a talented driver for Ferrari. Known for his qualifying speed and precision on the track, he has shown great promise in multiple seasons.',
  'George Russell drives for Mercedes alongside Lewis Ha